<a href="https://colab.research.google.com/github/iiiiiiiiice/dr1/blob/main/Diabetic_Retinopathy_cbam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q split-folders
import splitfolders
import os

input_data = '/content/processed_data'
output_data = '/content/Diabetic_Balanced_Data'

if not os.path.exists(output_data):
    os.makedirs(output_data)

# 检查输入文件夹是否存在以防止报错
if not os.path.exists(input_data):
    print(f"错误：输入文件夹 '{input_data}' 不存在。请检查您的数据集是否已解压并存放在该路径下。")
elif len(os.listdir(output_data)) == 0:
    splitfolders.ratio(input_data, output=output_data, seed=100, ratio=(.7, .2, .1), group_prefix=None)
    print("数据集拆分完成。")
else:
    print("输出文件夹非空，跳过拆分操作。")

错误：输入文件夹 '/content/processed_data' 不存在。请检查您的数据集是否已解压并存放在该路径下。


In [ ]:
!pip install -U keras-tuner
import keras_tuner as kt
print(f'Keras Tuner version: {kt.__version__}')
!rm -rf /content/sample_data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 7.9 MB/s eta 0:00:00
Keras Tuner version: 1.4.8


In [ ]:
from google.colab import drive
import os

# Robust check to avoid errors if already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted.')

Mounted at /content/drive


In [ ]:
import kagglehub
path = kagglehub.dataset_download("alisalen/diabetic-balanced-data-zip")

100%|██████████| 1.90G/1.90G [00:51<00:00, 39.5MB/s]

Extracting files...


In [ ]:
import os

print("Dataset downloaded to:", path)

# List the contents of the downloaded directory
print("\nContents of the downloaded dataset:")
for item in os.listdir(path):
    item_path = os.path.join(path, item)
    if os.path.isdir(item_path):
        print(f"[DIR]  {item}")
    else:
        print(f"[FILE] {item}")


Dataset downloaded to: /root/.cache/kagglehub/datasets/alisalen/diabetic-balanced-data-zip/versions/1

Contents of the downloaded dataset:
[DIR]  Diabetic_Balanced_Data


In [ ]:
import shutil
import os

source_dir = os.path.join(path, 'Diabetic_Balanced_Data')
target_dir = '/content/Diabetic_Balanced_Data'

if os.path.exists(source_dir):
    print(f"Copying dataset from {source_dir} to {target_dir}...")
    # Use dirs_exist_ok=True to copy into the existing directory
    shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)
    print("Copy completed!")
else:
    print(f"Source directory {source_dir} not found.")

# Let's list what's inside the target directory
print("\nContents of /content/Diabetic_Balanced_Data:")
!ls -l /content/Diabetic_Balanced_Data


Copying dataset from /root/.cache/kagglehub/datasets/alisalen/diabetic-balanced-data-zip/versions/1/Diabetic_Balanced_Data to /content/Diabetic_Balanced_Data...
Copy completed!

Contents of /content/Diabetic_Balanced_Data:
total 12
drwxr-xr-x 7 root root 4096 Jun  4 07:45 test
drwxr-xr-x 7 root root 4096 Jun  4 07:45 train
drwxr-xr-x 7 root root 4096 Jun  4 07:45 valid


```markdown
# Downloading The Dataset
```

In [ ]:
# 确保解压到根目录并确认输出文件夹名
!unzip -o /content/Diabetic_Balanced_Data.zip -d /content/
# 列出当前目录结构以排查嵌套问题
!find /content -maxdepth 3 -type d

In [ ]:
import os
import gc
import cv2
import glob
import random
import numpy as np
import pandas as pd
from os import path
from tqdm import tqdm
import seaborn as sns
import tensorflow as tf
from google.colab import drive
from google.colab import files
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam , SGD , RMSprop
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.resnet_v2 import ResNet50V2
from tensorflow.keras.applications.inception_v3 import InceptionV3

In [ ]:
def show_data(path_dataset):
  images_data = glob.glob(path_dataset)
  random.shuffle(images_data)
  plt.figure(figsize=(10,10))
  for i in range(9):
    plt.subplot(3,3,i+1)
    img = plt.imread(images_data[i-1])
    plt.imshow(img)

def plot_data(dataset):
  print('Total Number Of Images {}'.format(len(dataset)))
  img_files = [os.path.basename(name) for name in dataset]
  data_label = [str(name.split('/')[-2]) for name in dataset]
  df = pd.DataFrame({'filename':img_files,'label':data_label})
  sns.countplot(x=df['label'])
  df['label'].value_counts()
  return df

In [ ]:
# dataset = glob.glob('/content/content/processed_data/*/*.jpeg')
# df = plot_data(dataset)

In [ ]:
classes=['No_Dr','Mild','Moderate','severe','Proliferative DR']
class_dict = {}
for i,label in enumerate(classes):
  class_dict[i]=label
print(class_dict)

# label_1 = glob.glob('/content/processed_data/4/*.jpeg')
# label_1 = list(label_1[10000::])
# print(len(label_1))
# for i in range(len(label_1)):
#   os.remove(label_1[i])

{0: 'No_Dr', 1: 'Mild', 2: 'Moderate', 3: 'severe', 4: 'Proliferative DR'}


# Plot Images In A Directory -> Function

## Image Aug using IMAGAUG

In [ ]:
import glob
import pandas as pd
import os

# 采用递归搜索，解决嵌套文件夹路径问题
search_pattern = '/content/**/train/*/*.jpeg'
dataset = glob.glob(search_pattern, recursive=True)
print(f'Total Number Of Images found: {len(dataset)}')

if len(dataset) > 0:
    img_paths = dataset
    img_files = [os.path.basename(name) for name in dataset]
    # 提取倒数第二个目录名作为标签
    data_label = [int(name.split(os.sep)[-2]) for name in dataset]
    df = pd.DataFrame({'abs_path': img_paths, 'filename': img_files, 'label': data_label})
    print("Class Distribution:\n", df['label'].value_counts())
else:
    df = pd.DataFrame(columns=['abs_path', 'filename', 'label'])
    print("Warning: No images found. Check if unzip was successful or the search pattern is correct.")

Total Number Of Images found: 0


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import gc

# 使用绝对路径过滤少数类，避免拼接路径出错
if not df.empty:
    df_minor = df.loc[~df['label'].isin([0, 1, 2])]
    diabetic_imgs = df_minor['abs_path'].values
    print(f"Found {len(diabetic_imgs)} images for augmentation.")
    if len(diabetic_imgs) > 0:
        np.random.shuffle(diabetic_imgs)
else:
    diabetic_imgs = np.array([])
    print("DataFrame is empty, cannot proceed with augmentation.")

gc.collect()

# 修正 Keras 3 预处理层
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation((0.1, 0.3), fill_mode='nearest'),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])

DataFrame is empty, cannot proceed with augmentation.


In [ ]:
# 1. 强制安装 NumPy 1.x 以兼容 imgaug
!pip install --force-reinstall "numpy<2.0" imgaug

import os
import gc
import warnings
import imageio
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from google.colab.patches import cv2_imshow
from tensorflow.keras.preprocessing import image

# 检查 NumPy 版本，如果是 2.x 则提醒重启
if np.__version__.startswith('2.'):
    raise RuntimeError("NumPy 版本仍为 2.x。请点击『代码执行程序』->『重新启动会话』，然后再次运行此单元格。")

# 此时导入 imgaug 应该不会报错了
from imgaug import augmenters as iaa

warnings.filterwarnings('ignore')

seq = iaa.Sequential([
    iaa.Crop(px=(0, 16)),
    iaa.Fliplr(0.5),
    iaa.Affine(rotate=(-25,25)),
    iaa.LinearContrast(alpha=1.2),
    iaa.GaussianBlur(sigma=1.5)
])

def read_img(filename,shape=(512,512)):
    img = image.load_img(filename,target_size=shape)
    img = image.img_to_array(img)
    return img

def load_batch(img_list,batch=32,count=0):
    imgs = []
    fnames = []
    i = count*batch
    for filename in img_list[i:batch*(count+1)]:
        img = read_img(filename)
        imgs.append(img)
        fnames.append(filename)
    return imgs,fnames

# 检查 diabetic_imgs 是否存在，若为空则重新扫描路径
if 'diabetic_imgs' not in locals() or len(diabetic_imgs) == 0:
    import glob
    # 尝试匹配实际解压后的少数类路径 (1, 3, 4)
    diabetic_imgs = []
    for c in ['1', '3', '4']:
        diabetic_imgs.extend(glob.glob(f'/content/Diabetic_Balanced_Data/train/{c}/*.jpeg'))
    diabetic_imgs = np.array(diabetic_imgs)

if len(diabetic_imgs) > 0:
    nb_batches = len(diabetic_imgs)//32
    for idx in tqdm(range(nb_batches), position=0):
        images, fnames = load_batch(diabetic_imgs, count=idx)
        images_aug = seq(images=images)
        for im, im_aug in enumerate(images_aug):
            name = fnames[im][:-5]+'_aug_'+str(im)+'.jpeg'
            imageio.imwrite(name, im_aug)
    print(f"Successfully augmented images.")
else:
    print("No images found for augmentation in /content/Diabetic_Balanced_Data/train/.")

gc.collect()

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: NumPy 版本仍为 2.x。请点击『代码执行程序』->『重新启动会话』，然后再次运行此单元格。

In [ ]:
import numpy as np
print(f"当前 NumPy 版本: {np.__version__}")

当前 NumPy 版本: 2.0.2


In [ ]:
# 1. 安装/更新 Albumentations 库（Colab 通常已默认安装，此处确保最新）
!pip install -U albumentations

import os
import gc
import warnings
import imageio
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from google.colab.patches import cv2_imshow
from tensorflow.keras.preprocessing import image
import albumentations as A  # 导入 Albumentations 代替 imgaug

warnings.filterwarnings('ignore')

# 使用 Albumentations 重构您的增强序列 (seq)
# 对应关系：
# - iaa.Crop(px=(0, 16)) 和 iaa.Affine(rotate=(-25,25)) 整合为 A.Affine
# - iaa.Fliplr(0.5) 对应 A.HorizontalFlip(p=0.5)
# - iaa.LinearContrast(alpha=1.2) 对应 A.RandomBrightnessContrast
# - iaa.GaussianBlur(sigma=1.5) 对应 A.GaussianBlur
seq = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(
        translate_percent={"x": (-0.03, 0.03), "y": (-0.03, 0.03)}, # 类似轻微裁剪/平移效果
        rotate=(-25, 25),
        mode=0, # 边界填充模式
        p=1.0
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0,
        contrast_limit=(0.2, 0.2), # 固定增强对比度
        p=1.0
    ),
    A.GaussianBlur(blur_limit=(3, 5), p=1.0) # 模糊效果
])

def read_img(filename, shape=(512,512)):
    img = image.load_img(filename, target_size=shape)
    img = image.img_to_array(img)
    # Albumentations 通常需要 uint8 格式的数据
    return img.astype(np.uint8)

def load_batch(img_list, batch=32, count=0):
    imgs = []
    fnames = []
    i = count*batch
    for filename in img_list[i:batch*(count+1)]:
        img = read_img(filename)
        imgs.append(img)
        fnames.append(filename)
    return imgs, fnames

# 检查 diabetic_imgs 是否存在，若为空则重新扫描路径
if 'diabetic_imgs' not in locals() or len(diabetic_imgs) == 0:
    import glob
    diabetic_imgs = []
    for c in ['1', '3', '4']:
        diabetic_imgs.extend(glob.glob(f'/content/Diabetic_Balanced_Data/train/{c}/*.jpeg'))
    diabetic_imgs = np.array(diabetic_imgs)

if len(diabetic_imgs) > 0:
    nb_batches = len(diabetic_imgs)//32
    for idx in tqdm(range(nb_batches), position=0):
        images, fnames = load_batch(diabetic_imgs, count=idx)

        # 逐张应用 Albumentations 增强（因其接口针对单张图，亦可处理批次）
        images_aug = []
        for img in images:
            augmented = seq(image=img)
            images_aug.append(augmented['image'])

        for im, im_aug in enumerate(images_aug):
            name = fnames[im][:-5]+'_aug_'+str(im)+'.jpeg'
            imageio.imwrite(name, im_aug)
    print(f"Successfully augmented images.")
else:
    print("No images found for augmentation in /content/Diabetic_Balanced_Data/train/.")

gc.collect()

No images found for augmentation in /content/Diabetic_Balanced_Data/train/.


60

***Spliting Dataset***

In [ ]:
import os

# Unzip the provided dataset into the expected input directory
zip_path = '/content/diabetic.zip'
extract_path = '/content/processed_data'

if os.path.exists(zip_path):
    !unzip -q -o {zip_path} -d {extract_path}
    print("Dataset successfully unzipped to", extract_path)
else:
    print(f"File {zip_path} not found. Please ensure it is uploaded.")

[/content/diabetic.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/diabetic.zip or
        /content/diabetic.zip.zip, and cannot find /content/diabetic.zip.ZIP, period.
Dataset successfully unzipped to /content/processed_data


In [ ]:
import os

zip_path = '/content/diabetic.zip'

if os.path.exists(zip_path):
    # 获取文件大小 (MB)
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"当前 '{zip_path}' 的文件大小: {size_mb:.2f} MB")
    print("\n正在测试 zip 压缩包的完整性...")

    # 运行 zip 测试命令
    # -t: test archive data, -q: quiet
    !unzip -tq {zip_path}
else:
    print(f"文件 {zip_path} 不存在，请检查左侧面板确认是否开始上传。")


当前 '/content/diabetic.zip' 的文件大小: 32.00 MB

正在测试 zip 压缩包的完整性...
[/content/diabetic.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/diabetic.zip or
        /content/diabetic.zip.zip, and cannot find /content/diabetic.zip.ZIP, period.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q split-folders
import splitfolders
import os

input_data = '/content/processed_data'
output_data = '/content/Diabetic_Balanced_Data'

if not os.path.exists(output_data):
    os.makedirs(output_data)

# 检查输入文件夹是否存在以防止报错
if not os.path.exists(input_data):
    print(f"错误：输入文件夹 '{input_data}' 不存在。请检查您的数据集是否已解压并存放在该路径下。")
elif len(os.listdir(output_data)) == 0:
    splitfolders.ratio(input_data, output=output_data, seed=100, ratio=(.7, .2, .1), group_prefix=None)
    print("数据集拆分完成。")
else:
    print("输出文件夹非空，跳过拆分操作。")

错误：输入文件夹 '/content/processed_data' 不存在。请检查您的数据集是否已解压并存放在该路径下。


In [ ]:
from google.colab import drive
# 使用 force_remount=True 强制重新挂载，解决挂载卡死或失败的问题
drive.mount('/content/drive', force_remount=True)


In [ ]:
# !zip -r /content/Diabetic_Balanced_Data.zip /content/
!mv /content/Diabetic_Balanced_Data.zip /content/drive/MyDrive/

```markdown
#Model Creation
```

In [ ]:
# 统一路径，移除多余的 /content/content
IMG_WIDTH = 256
IMG_HEIGHT = 256
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT)
test_data = '/content/Diabetic_Balanced_Data/test'
training_data = '/content/Diabetic_Balanced_Data/train'
# Kaggle dataset uses 'valid' instead of 'val'
validation_data = '/content/Diabetic_Balanced_Data/valid'


In [ ]:
image_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(
  rescale = 1.0/255.0,
  )

training_datagen = image_data_generator.flow_from_directory(
    training_data,
    target_size=IMG_SHAPE,
    shuffle=True,
)

validation_datagen = image_data_generator.flow_from_directory(
    validation_data,
    target_size=IMG_SHAPE,
    shuffle = True
)

test_datagen = image_data_generator.flow_from_directory(
    test_data,
    target_size=IMG_SHAPE,
    shuffle=True)

# KERAS HYPERTUNER

In [ ]:
# !rm -rf /content/drive/MyDrive/Diabetic_Hypertuner

In [ ]:
import os
import keras_tuner as kt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.resnet_v2 import ResNet50V2

# 1. 自动检测数据路径 (防止嵌套文件夹导致找不到路径)
possible_paths = [
    '/content/Diabetic_Balanced_Data/train',
    '/content/content/Diabetic_Balanced_Data/train',
    '/content/train'
]

training_data = None
for p in possible_paths:
    if os.path.exists(p):
        training_data = p
        # Kaggle dataset uses 'valid' instead of 'val'
        validation_data = p.replace('/train', '/valid')
        print(f"Detected data path: {training_data}")
        break

if not training_data:
    print("Current files in /content:", os.listdir('/content'))
    raise FileNotFoundError("无法找到训练集文件夹，请检查解压路径。")

# 2. 确保必要的超参数已定义
IMG_WIDTH, IMG_HEIGHT = 256, 256
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT)
classes = ['No_Dr', 'Mild', 'Moderate', 'severe', 'Proliferative DR']

# 3. 定义数据生成器
image_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0/255.0)

training_datagen = image_data_generator.flow_from_directory(
    training_data, target_size=IMG_SHAPE, batch_size=32, class_mode='categorical', shuffle=True)

validation_datagen = image_data_generator.flow_from_directory(
    validation_data, target_size=IMG_SHAPE, batch_size=32, class_mode='categorical', shuffle=True)

# 4. 模型构建函数
def model_builder(hp):
    base_model = ResNet50V2(input_shape=(IMG_WIDTH, IMG_HEIGHT, 3), include_top=False, weights='imagenet')
    for layer in base_model.layers[:45]:
        layer.trainable = True
    x = tf.keras.layers.GlobalMaxPooling2D()(base_model.output)
    x = tf.keras.layers.Flatten()(x)
    hp_units = hp.Int('units', min_value=1300, max_value=1750, step=150)
    x = tf.keras.layers.Dense(hp_units, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    prediction_layer = tf.keras.layers.Dense(5, activation='softmax')(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=prediction_layer)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# 5. 初始化 Tuner 并开始搜索
tuner = kt.Hyperband(model_builder, objective='val_accuracy', max_epochs=10, factor=5,
                     directory='/content/drive/MyDrive/Diabetic_Hypertuner', project_name='diabetic_parameters')

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

tuner.search(training_datagen, epochs=5, verbose=1, validation_data=validation_datagen, callbacks=[stop_early])

Trial 1 Complete [00h 00m 05s]

Best val_accuracy So Far: None
Total elapsed time: 00h 00m 05s

Search: Running Trial #2

Value             |Best Value So Far |Hyperparameter
1300              |1450              |units
0.01              |0.001             |learning_rate
2                 |2                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
1                 |1                 |tuner/bracket
0                 |0                 |tuner/round



KeyboardInterrupt: 

In [ ]:
best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Hyperparameters: {best_hps}")

model = tuner.hypermodel.build(best_hps)
history = model.fit(training_datagen, epochs=20, validation_data=validation_datagen)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % (best_epoch,))

In [ ]:
!rm -rf /content/drive/MyDrive/Diabetic_Weight.h5

In [ ]:
diab_model.save('/content/drive/MyDrive/diab_model.h5')

In [ ]:
import tensorflow as tf
tf.saved_model.save(diab_model,'/content/drive/MyDrive/Diabetic_Weight.h5')

# Model

In [ ]:
%load_ext tensorboard
from datetime import datetime
import os

In [ ]:
def define_model(n_layers=45,BASE_MODEL='ResNet50V2'):
    if BASE_MODEL =='ResNet50V2':
        # Pre-trained model with ResNet50V2
        base_model = ResNet50V2(input_shape=(IMG_WIDTH,IMG_HEIGHT,3),include_top=False,weights='imagenet')
        for layer in base_model.layers[:n_layers]:
            layer.trainable=True
        head_model = base_model.output
        head_model = tf.keras.layers.GlobalMaxPooling2D()(head_model)
        head_model = tf.keras.layers.Flatten(name="Flatten")(head_model)
        head_model = tf.keras.layers.Dense(1600,activation='relu')(head_model)
        head_model = tf.keras.layers.Dropout(0.2)(head_model)
        prediction_layer = tf.keras.layers.Dense(len(classes), activation='softmax')(head_model)
        model = tf.keras.Model(inputs=base_model.input,outputs=prediction_layer)

    if BASE_MODEL =='InceptionV3':
        base_model = InceptionV3(input_shape=(IMG_WIDTH,IMG_HEIGHT,3),include_top=False,weights='imagenet')
        for layer in base_model.layers[:n_layers]:
            layer.trainable=False

        head_model = base_model.output
        head_model = tf.keras.layers.GlobalMaxPooling2D()(head_model)
        head_model = tf.keras.layers.Flatten(name="Flatten")(head_model)
        head_model = tf.keras.layers.Dense(1024,activation='relu')(head_model)
        head_model = tf.keras.layers.Dropout(0.5)(head_model)
        prediction_layer = tf.keras.layers.Dense(len(classes), activation='softmax')(head_model)
        model = tf.keras.Model(inputs=base_model.input,outputs=prediction_layer)
    return model

# define Model
model= define_model(BASE_MODEL='ResNet50V2')

#Compilation of the model
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    metrics=['accuracy'])

In [ ]:
# 修正 Keras 3 要求：使用 .weights.h5 后缀
checkpoint_path = "/content/drive/MyDrive/Custom_Weights.weights.h5"

cp_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    mode='min'
)

learning_rate_reduction = ReduceLROnPlateau(
    monitor='val_accuracy',
    patience=2,
    verbose=1,
    factor=0.3,
    min_lr=0.00001
)

logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)

In [ ]:
%tensorboard --logdir logs/scalars
history = model.fit(
    training_datagen,
    epochs=12,
    steps_per_epoch=1000,
    shuffle=True,
    validation_data=validation_datagen,
    callbacks=[cp_callback,learning_rate_reduction,tensorboard_callback])

In [ ]:
import gc
gc.collect()

In [ ]:
import tensorflow as tf
tf.saved_model.save(model,'/content/drive/MyDrive/Diabetic_Weight')

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from os.path import join
import numpy as np
import tensorflow as tf

diab_model = load_model('/content/drive/MyDrive/diab_model.h5')
shape = (256,256)

def decode_img(image_path, shape):
    # Note: original code used 'filename', updated to use 'image_path' for consistency
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=shape)
    img = tf.keras.preprocessing.image.img_to_array(img)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    return img

In [ ]:
import glob
import random
import matplotlib.pyplot as plt

test_img = glob.glob('/content/content/Diabetic_Balanced_Data/test/*/*.jpeg')
img_select = random.randint(0, len(test_img) - 1)

print(f"Selected image: {test_img[img_select]}")
img = plt.imread(test_img[img_select])
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
test_data = glob.glob('/content/content/Diabetic_Balanced_Data/test/*/*.jpeg')
print("Test data ", len(test_data))
img_files = [os.path.basename(name) for name in test_data]
test_label = [name.split('/')[-2] for name in test_data]
test_df = pd.DataFrame({'filename': img_files, 'label': test_label})
test_df.to_csv('test_data.csv')
display(test_df)

In [ ]:
predictions = []
for iter, row in test_df.iterrows():
    # Construct the full path using class label and filename
    filename = join('/content/content/Diabetic_Balanced_Data/test/', join(row.label, row.filename))
    # Pre-process image
    img = decode_img(filename, shape)
    # Get model prediction
    pred = diab_model.predict(img)
    y_classes = np.argmax(pred)
    # Store the raw prediction probabilities
    predictions.append(pred)


In [ ]:
test_df['pred_label'] = predictions
print(f"First prediction raw output: {predictions[0]}")
display(test_df.head())

In [ ]:
y_test = test_df['label'].astype(int)
y_pred = test_df['pred_label']

In [ ]:
y_pred = test_df.apply(lambda row: np.argmax(list(row['pred_label'])), axis=1)
print("True labels (y_test):")
print(y_test.values)

In [ ]:
import itertools
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Generate and print the classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Calculate the confusion matrix
cnf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion matrix calculated.")

In [ ]:
def plot_confusion_matrix(cm, classes, title='Confusion matrix', cmap=plt.cm.Blues):
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(10,10))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

np.set_printoptions(precision=2)

# plot normalized confusion matrix
plot_confusion_matrix(cnf_matrix, classes=classes, title='Normalized confusion matrix')
plt.show()

## Saving the Tuner's Best Model

In [ ]:
import tensorflow as tf
import os
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras.applications.resnet_v2 import ResNet50V2

tuner_best_model_save_path = '/content/drive/MyDrive/tuner_best_model.h5'

# Ensure required variables for tuner and model are available
# These values are taken from cells WNIA-z-VebFd and cs05sc2ACf7f
if 'IMG_WIDTH' not in locals():
    IMG_WIDTH = 256
if 'IMG_HEIGHT' not in locals():
    IMG_HEIGHT = 256
if 'classes' not in locals():
    classes = ['No_Dr', 'Mild', 'Moderate', 'severe', 'Proliferative DR']

# Re-define model_builder if not already defined (from XEQga0_xohg8)
if 'model_builder' not in locals():
    def model_builder(hp):
        base_model = ResNet50V2(input_shape=(IMG_WIDTH, IMG_HEIGHT, 3), include_top=False, weights='imagenet')
        for layer in base_model.layers[:45]:
            layer.trainable = True
        x = tf.keras.layers.GlobalMaxPooling2D()(base_model.output)
        x = tf.keras.layers.Flatten()(x)
        hp_units = hp.Int('units', min_value=1300, max_value=1750, step=150)
        x = tf.keras.layers.Dense(hp_units, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.3)(x)
        prediction_layer = tf.keras.layers.Dense(len(classes), activation='softmax')(x)

        model_builder_instance = tf.keras.Model(inputs=base_model.input, outputs=prediction_layer)
        hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
        model_builder_instance.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                    loss='categorical_crossentropy', metrics=['accuracy'])
        return model_builder_instance

# Re-establish tuner and get the best model if they are not in scope
if 'tuner' not in locals():
    print("Attempting to load Keras Tuner from disk.")
    tuner_dir = '/content/drive/MyDrive/Diabetic_Hypertuner'
    project_name = 'diabetic_parameters'
    try:
        # Load the tuner, overwrite=False to load existing results
        tuner = kt.Hyperband(model_builder, objective='val_accuracy', max_epochs=10, factor=5,
                             directory=tuner_dir, project_name=project_name, overwrite=False)
        print("Keras Tuner loaded successfully.")
    except Exception as e:
        print(f"Error loading Keras Tuner: {e}")
        tuner = None # Indicate failure

model = None
if tuner:
    try:
        best_models = tuner.get_best_models(num_models=1)
        if best_models:
            # Get the best trained model directly from the tuner
            # This will rebuild the model and load its best weights found during tuning
            model = best_models[0]
            print("Best model from Keras Tuner retrieved successfully.")
            tf.saved_model.save(model, tuner_best_model_save_path)
            print(f"Best model from Keras Tuner saved to: {tuner_best_model_save_path}")
        else:
            print("No best models found by the tuner. Please ensure Keras Tuner search (cell XEQga0_ohg8) was completed successfully.")
    except Exception as e:
        print(f"Error retrieving or saving the best model from tuner: {e}")
        print("Please ensure Keras Tuner search (cell XEQga0_ohg8) and model training (cell DxhxAVtYOgE-) were completed.")
else:
    print("Error: Keras Tuner object could not be loaded or re-initialized. Cannot save model.")


## Evaluating the Tuner's Best Model

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os # Added import for os.path.exists

# Ensure required variables for data generators are available
# These values are taken from cells cs05sc2ACf7f and K3JR6z_5GVjQ
if 'IMG_WIDTH' not in locals():
    IMG_WIDTH = 256
if 'IMG_HEIGHT' not in locals():
    IMG_HEIGHT = 256
if 'IMG_SHAPE' not in locals():
    IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT)

# Correctly set test_data path
test_data = '/content/Diabetic_Balanced_Data/test'
if not os.path.exists(test_data):
    print(f"Warning: 'test_data' path not found at {test_data}. Please ensure the dataset is unzipped correctly.")
    test_datagen = None # Indicate failure if path is wrong
else:
    # Ensure test_datagen is available, if not, recreate it
    if 'test_datagen' not in locals() or not isinstance(test_datagen, tf.keras.preprocessing.image.DirectoryIterator):
        print("Re-creating ImageDataGenerator and test_datagen.")
        if 'image_data_generator' not in locals():
            image_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0/255.0)
        try:
            test_datagen = image_data_generator.flow_from_directory(
                test_data,
                target_size=IMG_SHAPE,
                shuffle=True
            )
            print(f"Successfully re-created test_datagen with {test_datagen.n} images.")
        except Exception as e:
            print(f"Error re-creating test_datagen: {e}")
            test_datagen = None # Indicate failure

# Ensure 'model' is available (should be from the previous modified cell 13cb9d87)
if 'model' not in locals() or model is None:
    print("Error: 'model' object (from Keras Tuner) not found. Cannot make predictions.")
    predictions_tuner_model = None
    tuner_results_df = None # Ensure df is not created if model is missing
elif test_datagen is None:
    print("Error: 'test_datagen' not found due to incorrect path. Cannot make predictions.")
    predictions_tuner_model = None
    tuner_results_df = None
else:
    print("Making predictions with the tuner's best model...")
    try:
        predictions_tuner_model = model.predict(test_datagen)

        # Get true labels from the test_datagen
        test_labels = test_datagen.classes
        class_indices = test_datagen.class_indices
        idx_to_class = {v: k for k, v in class_indices.items()}
        true_labels_names = [idx_to_class[label] for label in test_labels]

        # Convert predictions to class labels
        predicted_labels_indices = np.argmax(predictions_tuner_model, axis=1)
        predicted_labels_names = [idx_to_class[label_idx] for label_idx in predicted_labels_indices]

        # Create a DataFrame for comparison
        tuner_results_df = pd.DataFrame({
            'True_Label': true_labels_names,
            'Predicted_Label': predicted_labels_names
        })
        display(tuner_results_df.head())
        print("Predictions completed for the tuner's best model.")
    except Exception as e:
        print(f"Error during prediction: {e}")
        tuner_results_df = None


### Debugging Data Paths

In [ ]:
import os

print("Contents of /content:")
!ls -F /content

print("\nContents of /content/Diabetic_Balanced_Data/ (if exists):")
if os.path.exists('/content/Diabetic_Balanced_Data'):
    !ls -F /content/Diabetic_Balanced_Data
else:
    print("Directory /content/Diabetic_Balanced_Data does not exist.")

print("\nContents of /content/Diabetic_Balanced_Data/test/ (if exists):")
if os.path.exists('/content/Diabetic_Balanced_Data/test'):
    !ls -F /content/Diabetic_Balanced_Data/test
else:
    print("Directory /content/Diabetic_Balanced_Data/test does not exist.")

print("\nPlease review the paths. If 'test' is inside another subfolder, we'll need to update the `test_data` variable.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import itertools

if 'tuner_results_df' in locals():
    # Generate and print the classification report
    print("\nClassification Report for Tuner's Best Model:")
    print(classification_report(tuner_results_df['True_Label'], tuner_results_df['Predicted_Label']))

    # Calculate the confusion matrix
    cnf_matrix_tuner = confusion_matrix(tuner_results_df['True_Label'], tuner_results_df['Predicted_Label'])
    print("Confusion matrix calculated for Tuner's Best Model.")

    # Define classes for plotting (ensure they are ordered correctly, e.g., from class_indices)
    sorted_classes = sorted(test_datagen.class_indices.keys(), key=lambda x: test_datagen.class_indices[x])

    # Plot normalized confusion matrix
    def plot_confusion_matrix(cm, classes, title='Confusion matrix', cmap=plt.cm.Blues):
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        plt.figure(figsize=(10,10))
        plt.imshow(cm, interpolation='nearest', cmap=cmap)
        plt.title(title)
        plt.colorbar()
        tick_marks = np.arange(len(classes))
        plt.xticks(tick_marks, classes, rotation=45)
        plt.yticks(tick_marks, classes)

        fmt = '.2f'
        thresh = cm.max() / 2.
        for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
            plt.text(j, i, format(cm[i, j], fmt),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")

        plt.ylabel('True label')
        plt.xlabel('Predicted label')
        plt.tight_layout()

    np.set_printoptions(precision=2)
    plot_confusion_matrix(cnf_matrix_tuner, classes=sorted_classes, title='Normalized Confusion Matrix for Tuner\'s Best Model')
    plt.show()
else:
    print("Tuner's results DataFrame not found. Cannot generate evaluation metrics.")

In [ ]:
"""
train_full.py - 糖尿病视网膜病变分级完整训练脚本
修改版：适配 Colab 环境，直接读取预先拆分好的 train 和 valid 文件夹
"""

import argparse
import random
import time
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from PIL import Image

# 检查并安装缺失的库
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ImportError:
    import os
    os.system('pip install albumentations')
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

try:
    import cv2
except ImportError:
    cv2 = None
    print("警告: OpenCV 未安装，部分预处理（去噪/CLAHE）将被禁用")

# ----------------------------- 工具函数 ---------------------------------
def set_seed(seed: int = 42) -> None:
    """固定随机种子，保证可复现性"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def _read_image_rgb(path: str) -> np.ndarray:
    """读取图片并转为 RGB numpy 数组"""
    if cv2 is not None:
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            raise FileNotFoundError(f"无法读取图片: {path}")
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    else:
        img = Image.open(path).convert("RGB")
        return np.array(img)

# ----------------------------- 预处理（去噪/CLAHE/裁剪）-----------------
class PreprocessConfig:
    def __init__(self, apply_denoise: bool = True, apply_clahe: bool = True, apply_crop: bool = True):
        self.apply_denoise = apply_denoise
        self.apply_clahe = apply_clahe
        self.apply_crop = apply_crop

def preprocess_retina(img_rgb: np.ndarray, cfg: PreprocessConfig) -> np.ndarray:
    """眼底图像专用预处理：双边滤波 + CLAHE + ROI 裁剪"""
    if cv2 is None:
        return img_rgb
    img = img_rgb.copy()
    if cfg.apply_denoise:
        img = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)
    if cfg.apply_clahe:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l2 = clahe.apply(l)
        lab2 = cv2.merge((l2, a, b))
        img = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)
    if cfg.apply_crop:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            cnt = max(contours, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(cnt)
            pad = int(0.08 * max(w, h))
            x1 = max(0, x - pad)
            y1 = max(0, y - pad)
            x2 = min(img.shape[1], x + w + pad)
            y2 = min(img.shape[0], y + h + pad)
            img = img[y1:y2, x1:x2]
    return img

# ----------------------------- 数据增强与数据集 -------------------------
def get_augmentations(image_size: int, is_train: bool) -> A.Compose:
    """返回 albumentations 数据增强序列"""
    mean = (0.485, 0.456, 0.406)
    std  = (0.229, 0.224, 0.225)
    if is_train:
        return A.Compose([
            A.Resize(image_size, image_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.7),
            A.RandomBrightnessContrast(p=0.5),
            A.HueSaturationValue(p=0.5, hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10),
            # 修复 albumentations 版本更新导致的警告
            A.CoarseDropout(p=0.2, num_holes_range=(1, 8), hole_height_range=(1, max(1, image_size//10)), hole_width_range=(1, max(1, image_size//10))),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(image_size, image_size),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])

class DRDataset(Dataset):
    """眼底图像数据集，支持预处理和增强"""
    def __init__(self, samples: List[Tuple[str, int]], image_size: int, is_train: bool,
                 preprocess_cfg: PreprocessConfig):
        self.samples = samples
        self.image_size = image_size
        self.is_train = is_train
        self.preprocess_cfg = preprocess_cfg
        self.transform = get_augmentations(image_size, is_train)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        path, label = self.samples[idx]
        img = _read_image_rgb(path)
        img = preprocess_retina(img, self.preprocess_cfg)
        out = self.transform(image=img)
        x = out["image"]
        y = torch.tensor(label, dtype=torch.long)
        return x, y

# ----------------------------- CBAM 模块 --------------------------------
class ChannelAttention(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(in_channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, in_channels, kernel_size=1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out) * x

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.cat([avg_out, max_out], dim=1)
        attn = self.sigmoid(self.conv(attn))
        return attn * x

class CBAM(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        self.ca = ChannelAttention(in_channels, reduction)
        self.sa = SpatialAttention(kernel_size=7)

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x

class CBAMWrap(nn.Module):
    """将 ResNet 的单个 Bottleneck/Block 包裹 CBAM"""
    def __init__(self, block: nn.Module, in_channels: int, reduction: int = 16):
        super().__init__()
        self.block = block
        self.cbam = CBAM(in_channels, reduction)

    def forward(self, x):
        x = self.block(x)
        x = self.cbam(x)
        return x

def build_cbam_resnet(backbone: str, num_classes: int, pretrained: bool = True,
                      dropout: float = 0.2, cbam_stages: List[int] = [1,2,3,4],
                      cbam_reduction: int = 16) -> nn.Module:
    """构建 CBAM-ResNet 模型"""
    backbone = backbone.lower()
    if backbone == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        stage_channels = {1:64, 2:128, 3:256, 4:512}
    elif backbone == "resnet34":
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        model = models.resnet34(weights=weights)
        stage_channels = {1:64, 2:128, 3:256, 4:512}
    elif backbone == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        stage_channels = {1:256, 2:512, 3:1024, 4:2048}
    else:
        raise ValueError(f"不支持的 backbone: {backbone}")

    # 在指定 stage 的最后一层插入 CBAM
    for stage_id in cbam_stages:
        layer = getattr(model, f"layer{stage_id}")
        if len(layer) == 0: continue
        layer[-1] = CBAMWrap(layer[-1], in_channels=stage_channels[stage_id], reduction=cbam_reduction)

    in_features = model.fc.in_features
    if dropout > 0:
        model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    else:
        model.fc = nn.Linear(in_features, num_classes)
    return model

# ----------------------------- 评估函数 ----------------------------------
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device,
             num_classes: int) -> dict:
    """计算验证集上的准确率、QWK、各类别 F1 等"""
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        prob = torch.softmax(logits, dim=1)
        pred = torch.argmax(prob, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())
        y_prob.extend(prob.cpu().numpy())
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    # 准确率
    acc = (y_true == y_pred).mean()
    # 二次加权 Kappa
    from sklearn.metrics import cohen_kappa_score, f1_score
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    # 各类别 F1
    f1_per_class = f1_score(y_true, y_pred, labels=list(range(num_classes)), average=None, zero_division=0)
    f1_macro = f1_per_class.mean()
    result = {
        "accuracy": float(acc),
        "qwk": float(qwk),
        "f1_macro": float(f1_macro),
    }
    for c in range(num_classes):
        result[f"f1_class_{c}"] = float(f1_per_class[c])
    return result

# ----------------------------- 主训练流程 --------------------------------
def load_data_from_folder(data_root: str) -> Tuple[List, List]:
    """
    从单个目录加载样本
    """
    data_root = Path(data_root)
    if not data_root.exists():
        raise FileNotFoundError(f"未找到目录: {data_root}")

    class_dirs = sorted([d for d in data_root.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_dirs]
    label_map = {name: idx for idx, name in enumerate(class_names)}

    samples = []
    for class_dir in class_dirs:
        label = label_map[class_dir.name]
        for img_path in class_dir.glob("*"):
            if img_path.suffix.lower() in ['.png','.jpg','.jpeg','.bmp']:
                samples.append((str(img_path), label))
    return samples, class_names

def train_full(args):
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1. 分别加载已经划分好的 train 和 valid 数据
    print("正在加载数据...")
    train_samples, class_names = load_data_from_folder(args.train_dir)
    val_samples, _ = load_data_from_folder(args.val_dir)

    num_classes = len(class_names)
    print(f"类别: {class_names}")
    print(f"训练样本数: {len(train_samples)}, 验证样本数: {len(val_samples)}")

    # 2. 预处理配置
    preprocess_cfg = PreprocessConfig(
        apply_denoise=not args.no_denoise,
        apply_clahe=not args.no_clahe,
        apply_crop=not args.no_crop
    )

    # 3. 数据集与数据加载器
    train_ds = DRDataset(train_samples, args.image_size, is_train=True, preprocess_cfg=preprocess_cfg)
    val_ds   = DRDataset(val_samples,   args.image_size, is_train=False, preprocess_cfg=preprocess_cfg)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              num_workers=args.num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=args.batch_size, shuffle=False,
                              num_workers=args.num_workers, pin_memory=True)

    # 4. 类别权重（解决不平衡）
    train_labels = [lab for _, lab in train_samples]
    class_counts = np.bincount(train_labels, minlength=num_classes)
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.mean()
    # 可选：额外增强困难类别
    if num_classes >= 5:
        class_weights[3] *= 2.5   # Severe
        class_weights[4] *= 3.0   # Proliferate
        class_weights = class_weights / class_weights.mean()
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    print("类别权重:", class_weights.cpu().numpy())

    # 5. 损失函数（支持 Focal Loss）
    if args.focal_loss:
        class FocalLoss(nn.Module):
            def __init__(self, alpha, gamma=2.0):
                super().__init__()
                self.alpha = alpha
                self.gamma = gamma
            def forward(self, inputs, targets):
                ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
                pt = torch.exp(-ce_loss)
                focal = ((1 - pt) ** self.gamma) * ce_loss
                return focal.mean()
        criterion = FocalLoss(alpha=class_weights, gamma=args.focal_gamma)
        print(f"使用 Focal Loss (gamma={args.focal_gamma})")
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        print("使用加权 CrossEntropyLoss")

    # 6. 构建模型
    model = build_cbam_resnet(
        backbone=args.backbone,
        num_classes=num_classes,
        pretrained=not args.no_pretrained,
        dropout=args.dropout,
        cbam_stages=[int(x) for x in args.cbam_stages.split(",")],
        cbam_reduction=args.cbam_reduction
    ).to(device)

    # 7. 优化器与调度器
    optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    # 8. 训练循环
    best_qwk = -1.0
    best_path = out_dir / "best_model.pth"
    print("\n开始训练...")
    print("=" * 70)

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            if args.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()

            total_loss += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)

        train_loss = total_loss / total
        train_acc = correct / total
        epoch_time = time.time() - start_time

        # 验证
        val_metrics = evaluate(model, val_loader, device, num_classes)
        qwk = val_metrics["qwk"]
        f1_3 = val_metrics.get("f1_class_3", 0)
        f1_4 = val_metrics.get("f1_class_4", 0)

        print(f"Epoch {epoch:3d}/{args.epochs} | "
              f"Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Time: {epoch_time:.1f}s")
        print(f"         Val Acc: {val_metrics['accuracy']:.4f} | QWK: {qwk:.4f} | "
              f"F1(3): {f1_3:.4f} | F1(4): {f1_4:.4f}")

        # 保存最佳模型（基于 QWK）
        if qwk > best_qwk:
            best_qwk = qwk
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "image_size": args.image_size,
                "backbone": args.backbone,
                "num_classes": num_classes,
                "qwk": qwk,
                "accuracy": val_metrics["accuracy"]
            }, best_path)
            print(f"  -> 保存最佳模型 (QWK={qwk:.4f})")
        scheduler.step()
        print("-" * 50)

    print("=" * 70)
    print(f"训练完成！最佳 QWK = {best_qwk:.4f}")
    print(f"最佳模型保存在: {best_path}")

# ----------------------------- 命令行参数 --------------------------------
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="糖尿病视网膜病变分级完整训练脚本")
    # 数据参数 - 适配 Colab 目录结构
    parser.add_argument("--train-dir", type=str,
                        default="/content/Diabetic_Balanced_Data/train",
                        help="训练集目录")
    parser.add_argument("--val-dir", type=str,
                        default="/content/Diabetic_Balanced_Data/valid",
                        help="验证集目录")
    parser.add_argument("--out-dir", type=str, default="/content/drive/MyDrive/checkpoints_cbam", help="模型保存目录（存到云盘防丢失）")

    # 模型参数
    parser.add_argument("--backbone", type=str, default="resnet50", choices=["resnet18","resnet34","resnet50"])
    parser.add_argument("--no-pretrained", action="store_true", help="不使用 ImageNet 预训练权重")
    parser.add_argument("--dropout", type=float, default=0.3, help="分类头 Dropout 比例")
    parser.add_argument("--cbam-stages", type=str, default="1,2,3,4", help="插入 CBAM 的 stage 编号，逗号分隔")
    parser.add_argument("--cbam-reduction", type=int, default=16, help="CBAM 通道缩减比例")

    # 训练参数
    parser.add_argument("--image-size", type=int, default=256, help="输入图像尺寸（根据显存调整）")
    parser.add_argument("--epochs", type=int, default=30, help="训练轮数")
    parser.add_argument("--batch-size", type=int, default=16, help="批次大小（根据显存调整）")
    parser.add_argument("--lr", type=float, default=1e-4, help="初始学习率")
    parser.add_argument("--weight-decay", type=float, default=1e-4, help="权重衰减")
    parser.add_argument("--grad-clip", type=float, default=1.0, help="梯度裁剪阈值")
    parser.add_argument("--num-workers", type=int, default=2, help="数据加载线程数")
    parser.add_argument("--seed", type=int, default=42, help="随机种子")

    # 损失函数增强
    # 默认开启 focal loss 以解决类别不平衡
    parser.add_argument("--focal-loss", action="store_true", default=True, help="使用 Focal Loss 缓解类别不平衡")
    parser.add_argument("--focal-gamma", type=float, default=2.0, help="Focal Loss 的 gamma 参数")

    # 预处理开关
    parser.add_argument("--no-denoise", action="store_true", help="禁用双边滤波去噪")
    parser.add_argument("--no-clahe", action="store_true", help="禁用 CLAHE 对比度增强")
    parser.add_argument("--no-crop", action="store_true", help="禁用 ROI 裁剪")

    # 使用 parse_known_args 避免在 Jupyter 环境中解析报错
    # 方案一：传入参数禁用耗时的预处理操作，并调大 batch_size 和 num_workers 以吃满 GPU
    args, _ = parser.parse_known_args([
        '--no-denoise',
        '--no-clahe',
        '--no-crop',
        '--batch-size', '64',
        '--num-workers', '4'
    ])
    train_full(args)


使用设备: cuda
正在加载数据...
类别: ['Mild', 'Moderate', 'No DR', 'Proliferative DR', 'Severe']
训练样本数: 41576, 验证样本数: 9940
类别权重: [0.32163206 0.6237824  0.6237824  1.559456   1.8713472 ]
使用 Focal Loss (gamma=2.0)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



开始训练...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch   1/30 | Loss: 0.5587 | Acc: 0.2755 | Time: 548.0s
         Val Acc: 0.4744 | QWK: 0.3412 | F1(3): 0.6405 | F1(4): 0.5411
  -> 保存最佳模型 (QWK=0.3412)
--------------------------------------------------


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch  12/30 | Loss: 0.0475 | Acc: 0.7951 | Time: 549.8s
         Val Acc: 0.7467 | QWK: 0.7779 | F1(3): 0.9788 | F1(4): 0.9470
  -> 保存最佳模型 (QWK=0.7779)
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch  13/30 | Loss: 0.0406 | Acc: 0.8119 | Time: 549.1s
         Val Acc: 0.7663 | QWK: 0.8096 | F1(3): 0.9836 | F1(4): 0.9581
  -> 保存最佳模型 (QWK=0.8096)
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
from google.colab import files
import os

model_path = '/content/drive/MyDrive/checkpoints_cbam/best_model.pth'

if os.path.exists(model_path):
    print(f"正在准备下载: {model_path}")
    files.download(model_path)
else:
    print(f"文件未找到: {model_path}\n请确保训练已经完成并且模型已经保存。")
